# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") via its Croissant schema using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is specified by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load and inspect dataset metadata with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Note: dataset.metadata is an object (not a dictionary)
print("Dataset Metadata:")
print(f"  Name: {dataset.metadata.name}")
print(f"  Description: {dataset.metadata.description}")
print(f"  Version: {dataset.metadata.version}")
print(f"  Date Published: {dataset.metadata.date_published if hasattr(dataset.metadata, 'date_published') else dataset.metadata.datePublished}")
print(f"  License: {dataset.metadata.license}")

## 2. Data Overview
Examine what record sets, fields, and columns are defined with their unique `@id`s.

**Note:** We use the `@id` value for all references. If you do not know what is present, the following code will enumerate available record sets, fields, and columns.

In [ ]:
# List available record sets and their fields/columns by @id
print("Record Sets (@id and name):")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs.id}\n    name: {getattr(rs, 'name', 'N/A')}")
    record_sets.append(rs.id)

    if hasattr(rs, 'fields'):
        print("    Fields (by @id, name, data_type):")
        for f in rs.fields:
            print(f"      - @id: {f.id} | name: {f.name} | data_type: {f.data_type}")
    if hasattr(rs, 'columns'):
        print("    Columns (by @id, name, data_type):")
        for c in rs.columns:
            print(f"      - @id: {c.id} | name: {c.name} | data_type: {c.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access is by the record set and field `@id` as shown above.

In [ ]:
# Choose a record set to extract data for. Replace this with the actual @id found above.
# For this notebook, we automatically take the first record set.
assert len(record_sets) > 0, "No record sets found in this dataset."
PRIMARY_RECORDSET_ID = record_sets[0]

print(f"\nExtracting records from record set: {PRIMARY_RECORDSET_ID}\n")
records = list(dataset.records(record_set=PRIMARY_RECORDSET_ID))
df = pd.DataFrame(records)

print(f"Columns returned from record set '{PRIMARY_RECORDSET_ID}':")
pprint(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filter for numeric criteria, normalize, and group by key features.

> **Instructions:** Replace variables below with actual field `@id`s appropriate for the dataset. You can copy them from the record set overview above.

In [ ]:
# Select a numeric field (by @id) for filtering/normalization (inspect column names printed above):
# For this dataset (mass spectrometry protein data), examples might include fields like 'cr:mw' or 'cr:coverage'
# Please change these as appropriate to your dataset - consult earlier cell outputs.

NUMERIC_FIELD = None
for col in df.columns:
    # Suggest use of molecular weight (MW) or any field with numeric data
    if 'mw' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower():
        NUMERIC_FIELD = col
        break
if NUMERIC_FIELD is None and len(df.select_dtypes('number').columns) > 0:
    NUMERIC_FIELD = df.select_dtypes('number').columns[0]
print(f"Numeric field for analysis: {NUMERIC_FIELD}")

# For group field, pick one with low cardinality (for protein datasets, often group by 'sample' or 'cr:sample')
GROUP_FIELD = None
for col in df.columns:
    if 'sample' in col.lower() or 'type' in col.lower() or 'category' in col.lower():
        GROUP_FIELD = col
        break

# EDA: Filtering, Normalizing, Grouping
if NUMERIC_FIELD is not None:
    threshold = df[NUMERIC_FIELD].mean() if df[NUMERIC_FIELD].dtype.kind in 'fc' else 10
    filtered_df = df[df[NUMERIC_FIELD] > threshold]
    print(f"Filtered records where '{NUMERIC_FIELD}' > {threshold:.2f} (using mean as threshold): {filtered_df.shape[0]} records.")
    
    # Normalization
    filtered_df = filtered_df.copy()
    filtered_df[f"{NUMERIC_FIELD}_normalized"] = (
        (filtered_df[NUMERIC_FIELD] - filtered_df[NUMERIC_FIELD].mean()) / filtered_df[NUMERIC_FIELD].std()
    )
    print(f"Show normalized '{NUMERIC_FIELD}' field:")
    print(filtered_df[[NUMERIC_FIELD, f"{NUMERIC_FIELD}_normalized"]].head())
    
    # Grouping (if group field available)
    if GROUP_FIELD is not None:
        grouped_df = filtered_df.groupby(GROUP_FIELD)[NUMERIC_FIELD].mean().to_frame(name=f"{NUMERIC_FIELD}_mean")
        print(f"Grouped mean of '{NUMERIC_FIELD}' by '{GROUP_FIELD}':")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA. Please inspect df.columns and specify an appropriate field.")

## 5. Visualization
Visualize distributions and relationships between numeric and categorical fields in the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if NUMERIC_FIELD is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[NUMERIC_FIELD].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {NUMERIC_FIELD}")
    plt.xlabel(NUMERIC_FIELD)
    plt.show()
    
    if GROUP_FIELD is not None:
        plt.figure(figsize=(12,5))
        sns.boxplot(data=df, x=GROUP_FIELD, y=NUMERIC_FIELD)
        plt.title(f"{NUMERIC_FIELD} by {GROUP_FIELD}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no numeric field selected.")

## 6. Conclusion
In this notebook, we:
- Loaded and inspected dataset metadata and structure from the FAIR^2 Croissant schema.
- Explored available record sets and fields, referencing all by their unique `@id`s.
- Extracted tabular data and performed basic exploratory analysis (filtering, normalization, grouping).
- Visualized the distribution of a key numeric field and its relationship with groupings in the data.

This workflow enables transparent, schema-driven data loading and analysis directly from the Croissant standard using `mlcroissant`.